In [50]:
import os
import time

from decorator import append
from dotenv import load_dotenv
from duckdb.experimental.spark.sql.functions import length

load_dotenv(override=True)

RIOT_PERSONAL_API_KEY = os.getenv("LOL_PERSONAL_API_KEY")
POSTGRES_LOCAL_URL = os.getenv("POSTGRES_LOCAL_URL")

In [18]:
import re

_camel_case_regex = re.compile(
    r"(?<=[a-z])(?=[A-Z])"  # camelCase: survive|Min
    r"|(?<=[A-Z])(?=[A-Z][a-z])"  # acronym: HTML|Parser
    r"|(?<=[a-zA-Z])(?=\d)"  # letter→digit: survive|15
    r"|(?<=\d)(?=[a-zA-Z])"  # digit→letter: 15|min
)


def camel_to_snake(input_str: str) -> str:
    return _camel_case_regex.sub("_", input_str).lower()


def snake_all_keys(json_obj: dict):
    if isinstance(json_obj, dict):
        return {camel_to_snake(key): snake_all_keys(value) for key, value in json_obj.items()}

    if isinstance(json_obj, list):
        return [snake_all_keys(value) for value in json_obj]

    return json_obj


In [89]:

import requests

BASE_EUN1_URL = "https://eun1.api.riotgames.com"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

# Fetch league entries (100 players)
ranks = ["IRON","BRONZE","SILVER","GOLD","PLATINUM","EMERALD","DIAMOND","MASTER","GRANDMASTER","CHALLENGER"]
tiers = ["I","II","III","IV"]
page = 1
league_entries = []
urls = []

for rank in ranks:
    for tier in tiers:
        if rank =="MASTER":
          urls.append(f"{BASE_EUN1_URL}/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5")
          break
        elif rank == "GRANDMASTER":
          urls.append(f"{BASE_EUN1_URL}/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5")
          break
        elif rank == "CHALLENGER":
          urls.append(f"{BASE_EUN1_URL}/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5")
          break
        else:
          urls.append(f"{BASE_EUN1_URL}/lol/league/v4/entries/RANKED_SOLO_5x5/{rank}/{tier}")
print(urls)



['https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/I', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/II', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/III', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/IV', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/II', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/III', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/IV', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/SILVER/I', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/SILVER/II', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/SILVER/III', 'https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/SILVER/IV', 'https://eun1.api.riotg

In [72]:
import polars as pl

pl.DataFrame(league_entries)

status
struct[2]
"{""Bad Request - invalid parameter value rank, must be one of [DIAMOND,EMERALD,PLATINUM,GOLD,SILVER,BRONZE,IRON]"",400}"


In [79]:

from datetime import datetime
import requests

BASE_EUNE_URL = "https://europe.api.riotgames.com"

headers = {
    'X-Riot-Token': RIOT_PERSONAL_API_KEY
}

def get_player_data_from_puuid(puuid: str):
    url = f"{BASE_EUNE_URL}/riot/account/v1/accounts/by-puuid/{puuid}"
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("request failed with status code", response.status_code)
        return None

    response_json = response.json()
    player_data = {
        "game_name": response_json["gameName"],
        "tag_line": response_json["tagLine"],
        "puuid": puuid,
        "synced_at": datetime.now(),
        "region": "EUN1"
    }

    return player_data


get_player_data_from_puuid(league_entries[0]["puuid"])
#response = requests.get(url, headers=headers, params={"page": page})
# league_entries = snake_all_keys(response.json())
# league_entries
# page = 1
# league_entries = []
# url = f"{BASE_EU_URL}/riot/account/v1/accounts/by-puuid/{}"
# response = requests.get(url, headers=headers, params={"page": page})
# league_entries = snake_all_keys(response.json())
# league_entries

{'game_name': 'EtA MetaliCator',
 'tag_line': 'EUNE',
 'puuid': 'zAkgQ_1WZ3Zq9h4lqhOaChgiWQmcRbowpXZAsYOn5rkUp2ZAtJmLwbwTMpIZ54Uqe_bC3Vflk-jN4w',
 'synced_at': datetime.datetime(2026, 5, 31, 17, 41, 47, 398461),
 'region': 'EUN1'}

In [93]:
player_data_arr = []
for rank_url in urls:
    print("now processing", rank_url)

    response = requests.get(rank_url, headers=headers)
    while response.status_code != 200:
         response = requests.get(rank_url, headers=headers)
         time.sleep(5)

    league_entries = snake_all_keys(response.json())
    if "entries" in league_entries:
        league_entries = league_entries["entries"]

    player_data_current_rank_array = []
    for entry in league_entries:
        player_data = get_player_data_from_puuid(entry["puuid"])

        if player_data is None:
            time.sleep(31)
            continue

        player_data_current_rank_array.append(player_data)
        if len(player_data_current_rank_array) >= 100:
            break

    player_data_arr += player_data_current_rank_array


now processing https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/I
request failed with status code 429
request failed with status code 429
now processing https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/II
request failed with status code 429
request failed with status code 429
request failed with status code 429
now processing https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/III
request failed with status code 429
request failed with status code 429
request failed with status code 429
now processing https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/IRON/IV
request failed with status code 429
request failed with status code 429
request failed with status code 429
now processing https://eun1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/BRONZE/I
request failed with status code 429
request failed with status code 429
request failed with status code 429
now processing https://eun1.api.rio

In [96]:
player_data_df = pl.DataFrame(player_data_arr)
player_data_df

game_name,tag_line,puuid,synced_at,region
str,str,str,datetime[μs],str
"""bird raven""","""0000""","""x1jqYLksFHSjj-V_dTeZrzR9fFEVDt…",2026-05-31 18:10:37.551072,"""EUN1"""
"""KaizokuNoOni""","""2829""","""QMPPN4oUrY-FW-BBOZ34yDZZmjghPa…",2026-05-31 18:10:37.860377,"""EUN1"""
"""Nolba""","""EUNE""","""yf1OVyzGtKFWKAh__NIUFtnEisdohM…",2026-05-31 18:10:38.176115,"""EUN1"""
"""Defenser""","""EUNE""","""gw0udTm33m9cDPfENXndpjlXmkHwxl…",2026-05-31 18:10:38.472933,"""EUN1"""
"""Dioklis""","""EUNE""","""tD8QYcsn9jmPaM7AACN9Id2fwgusfN…",2026-05-31 18:10:38.757716,"""EUN1"""
…,…,…,…,…
"""black nails""","""988""","""dYiEC-ravKg74XNQMp4KsOp4S7aVlR…",2026-05-31 19:15:39.289847,"""EUN1"""
"""AleEmotkaZostała""","""RMJ""","""ZLw_1J27yiyrFC_UN7-Uck-IyYMBLL…",2026-05-31 19:15:39.630180,"""EUN1"""
"""chiefelo""","""0314""","""0CPr6RzHVit2UWf0Cu0EXPbIj77eko…",2026-05-31 19:15:39.980448,"""EUN1"""


In [97]:
player_data_df.write_database(
    table_name="players",
    connection=POSTGRES_LOCAL_URL,
    if_table_exists="append",
    engine="adbc"
)

3100

3100
